# SimSat DiLoCo Round 0 Learner

Continues the DiLoCo global adapter (round 0, seeded from `simsat-gemma4-v10-adapter`) on the review-refreshed SimSat ChatML dataset (`benhaslam/simsat-gemma4-v1` v2 — 713 rows, REFINE_BOOST=1.5).

**Inputs (all attached):**
1. Model: `google/gemma-4` Transformers → `gemma-4-e2b-it/1`
2. Dataset: `benhaslam/simsat-gemma4-v1` (training JSONL)
3. Dataset: `benhaslam/diloco-lab-src` (DiLoCo source — auto-extracted by Kaggle)
4. Dataset: `benhaslam/diloco-global-round-000000` (round-0 global adapter — auto-extracted)

**Outputs:**
- `/kaggle/working/diloco_continued_adapter/` — continued LoRA + tokenizer + summary
- `/kaggle/working/diloco_outbox/<experiment>/round-000000/<learner>/` — fragment deltas

Sync + eval steps: see `D:\SimSat\notebooks\DILOCO_ROUND0_RUN_NOTE.md` in the SimSat repo.

In [ ]:
import subprocess, sys, zipfile
from pathlib import Path

INPUT = Path('/kaggle/input')

def _resolve_source(prefer_dir: str, zip_name: str, marker: str) -> Path:
    """Kaggle auto-extracts zips on dataset upload, but some setups still ship raw .zip.
    Prefer the extracted-dir form (look for `marker` file inside any /kaggle/input subtree);
    fall back to extracting `zip_name` to /kaggle/working if no extracted copy is found.
    Returns the directory that contains `marker`."""
    direct = list(INPUT.rglob(marker))
    if direct:
        return direct[0].parent
    zips = list(INPUT.rglob(zip_name))
    if not zips:
        raise RuntimeError(f'Neither extracted {marker} nor {zip_name} found under /kaggle/input. '
                           f'Make sure the dataset {prefer_dir!r} is attached.')
    target = Path(f'/kaggle/working/{prefer_dir}')
    if not target.exists():
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(target)
    return next(target.rglob(marker)).parent

# DiLoCo source bundle: contains kaggle/continue_gemma4_adapter.py + diloco_lab/ package
SRC = _resolve_source('diloco_lab_src', 'diloco_lab_source.zip', 'continue_gemma4_adapter.py').parent
print(f'SRC: {SRC}')

# Round-0 global adapter: contains adapter_config.json + adapter_model.safetensors + tokenizer.*
GLOBAL = _resolve_source('global_adapter', 'global_round_000000.zip', 'adapter_config.json')
print(f'GLOBAL: {GLOBAL}')

subprocess.check_call([
    sys.executable,
    str(SRC / 'kaggle' / 'continue_gemma4_adapter.py'),
    '--base-adapter', str(GLOBAL),
    '--project', 'simsat',
    '--dataset-id', 'simsat-gemma4-v3-reviewed',
    '--learner-id', 'kaggle-t4-simsat-round0-a',
    '--round-id', '0',
    '--max-steps', '120',
    '--lr', '5e-5',
])